# Spam Email Classifier - Exploratory Data Analysis & Model Building

This notebook walks through the complete ML workflow for the **Spam Email Classifier** project:

1. Data Loading
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Text Preprocessing & Tokenization
5. TF-IDF Vectorization
6. Train-Test Split
7. Model Training (Naive Bayes)
8. Model Evaluation (Accuracy, Precision, Recall, F1)
9. Confusion Matrix


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords', quiet=True)


## 1. Data Loading

In [ ]:
df = pd.read_csv('../dataset/spam_emails.csv')
print("Shape:", df.shape)
df.head()


## 2. Data Cleaning

In [ ]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Drop duplicates
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

# Encode labels: ham -> 0, spam -> 1
df['label'] = df['label'].str.strip().str.lower()
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
print(df['label'].value_counts())

plt.figure(figsize=(5,4))
df['label'].value_counts().plot(kind='bar', color=['#4CAF50', '#F44336'])
plt.title('Class Distribution: Ham vs Spam')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Message length analysis
df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(lambda x: len(x.split()))

print(df.groupby('label')[['num_characters', 'num_words']].describe())

plt.figure(figsize=(8,5))
plt.hist(df[df['label']=='ham']['num_characters'], bins=30, alpha=0.6, label='Ham')
plt.hist(df[df['label']=='spam']['num_characters'], bins=30, alpha=0.6, label='Spam')
plt.legend()
plt.title('Distribution of Message Length (Characters)')
plt.xlabel('Number of Characters')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


## 4. Text Preprocessing & Tokenization

In [ ]:
STOPWORDS = set(stopwords.words('english'))
STEMMER = PorterStemmer()

def preprocess_text(text):
    """Lowercase, remove URLs/punctuation/numbers, tokenize, remove stopwords, stem."""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', ' ', text)
    tokens = text.split()
    cleaned = [STEMMER.stem(w) for w in tokens if w not in STOPWORDS and len(w) > 1]
    return ' '.join(cleaned)

df['cleaned_text'] = df['text'].apply(preprocess_text)
df[['text', 'cleaned_text']].head(10)


## 5. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['cleaned_text']).toarray()
y = df['target'].values

print("Feature matrix shape:", X.shape)


## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


## 7. Model Training - Multinomial Naive Bayes

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)
print("Model trained successfully!")


## 8. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1 Score : {f1*100:.2f}%")

print("\n", classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))


## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Ham','Spam']); ax.set_yticklabels(['Ham','Spam'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14, fontweight='bold')

fig.colorbar(im)
plt.tight_layout()
plt.show()


## 10. Test with Custom Messages

In [ ]:
sample_messages = [
    "Congratulations! You have won a free iPhone. Click here to claim now!",
    "Hey, are we still meeting for lunch tomorrow at noon?",
]

for msg in sample_messages:
    cleaned = preprocess_text(msg)
    vec = tfidf.transform([cleaned]).toarray()
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    label = 'SPAM' if pred == 1 else 'HAM'
    print(f"Message: {msg}")
    print(f"Prediction: {label} | Confidence: {max(prob)*100:.2f}%\n")
